In [1]:
import numpy as np
import os,sys,glob
from pathlib import Path
import logging
from helper import (filterObjects,getModelInfo,saveOutput, \
                    electron_reco, muon_reco, deltaR, cutFlow, getD0)
from numpy import ndarray
from typing import Any, Dict, List, Tuple, Union
import multiprocessing
import subprocess
from computeEfficiencies import getSR,preSelection,getObjects
DelphesLLP_path = Path(os.path.abspath("./DelphesLLP"))
os.environ['ROOT_INCLUDE_PATH'] = os.path.join(DelphesLLP_path,"external")

import ROOT
ROOT.gSystem.Load(os.path.join(DelphesLLP_path,"libDelphes.so"))
ROOT.gInterpreter.Declare('#include "classes/SortableObject.h"')
ROOT.gInterpreter.Declare('#include "classes/DelphesClasses.h"')
ROOT.gInterpreter.Declare('#include "external/ExRootAnalysis/ExRootTreeReader.h"')
from ROOT import TFile,Electron, Jet, MissingET, Muon, TTree

Welcome to JupyROOT 6.30/06


### Cutflow from ATLAS:

In [15]:
# $\tilde{e}$ (mass, lifetime) = (100 GeV, 0.01 ns)
print(r'initial number of events ($\mathcal{L} \times \sigma$) =',50830.0,f'({50830.0/50830.0:1.3e})')
print(r'pass trigger and at least 2 baseline leptons =',736.0,f'({736.0/50830.0:1.3e})')
print(r'2 leading leptons are electrons =',393.0,f'({393.0/50830.0:1.3e})')
print(r'$p_\text{T} > 65$ GeV =',330.0,f'({330.0/50830.0:1.3e})')
print(r'$3$ mm$ < |d_{0}| < 300$ mm  =',121.0,f'({121.0/50830.0:1.3e})')
print(r'both electrons pass isolation =',117.0,f'({117.0/50830.0:1.3e})')
print(r'$(p_\text{T}^\text{track}-p_\text{T}^e)/p_\text{T}^e$ $\geq -0.5$ =',85.0,f'({85.0/50830.0:1.3e})')
print(r'ID track $\chi^2/n_{\mathrm{DOF}} <  2$ =',77.1,f'({77.1/50830.0:1.3e})')
print(r'number of missing layers $\leq 1$ =',77.1,f'({77.1/50830.0:1.3e})')
print(r'$\Delta R_{\ell\ell}$ $> 0.2$ =',77.1,f'({77.1/50830.0:1.3e})')
print(r'no additional cosmic muons =',77.1,f'({77.1/50830.0:1.3e})')

initial number of events ($\mathcal{L} \times \sigma$) = 50830.0 (1.000e+00)
pass trigger and at least 2 baseline leptons = 736.0 (1.448e-02)
2 leading leptons are electrons = 393.0 (7.732e-03)
$p_\text{T} > 65$ GeV = 330.0 (6.492e-03)
$3$ mm$ < |d_{0}| < 300$ mm  = 121.0 (2.380e-03)
both electrons pass isolation = 117.0 (2.302e-03)
$(p_\text{T}^\text{track}-p_\text{T}^e)/p_\text{T}^e$ $\geq -0.5$ = 85.0 (1.672e-03)
ID track $\chi^2/n_{\mathrm{DOF}} <  2$ = 77.1 (1.517e-03)
number of missing layers $\leq 1$ = 77.1 (1.517e-03)
$\Delta R_{\ell\ell}$ $> 0.2$ = 77.1 (1.517e-03)
no additional cosmic muons = 77.1 (1.517e-03)


In [ ]:
# sigma*eff_UL(SR_ee) = 0.02 fb
# sigma_UL(SR_ee) = 1.42e-2 pb (m = 100, tau = 0.01 ns)
eff = 0.02 / (1.42e-2*1e3)
print(eff)

0.001408450704225352


In [ ]:
inputFile = './pp2selselv2/Events/run_02/selectron_100GeV_0.010ns_delphes_events.root'
f = TFile(inputFile,'read')
DelphesTree = f.Get('Delphes')
nevts = DelphesTree.GetEntries()
print(nevts)

21491


In [4]:
entry = 1
DelphesTree.GetEntry(entry)

3844

In [5]:
llps,muons,electrons = getObjects(DelphesTree)

In [6]:
for llp in llps:
    print(f"LLP: PT={llp.PT}, Eta={llp.Eta}, Phi={llp.Phi}, Mass={llp.Mass}, Charge={llp.Charge}, PID = {llp.PID}")

LLP: PT=51.62721252441406, Eta=-2.5466556549072266, Phi=-2.8148560523986816, Mass=100.0, Charge=-1, PID = 1000011
LLP: PT=73.00959777832031, Eta=-2.6995701789855957, Phi=0.5278975367546082, Mass=100.0, Charge=1, PID = -1000011


In [7]:
daughters = DelphesTree.bsmDirectDaughters
for d in daughters:
    print(f"Daughter: PT={d.PT}, Eta={d.Eta}, Phi={d.Phi},  Charge={d.Charge}, PID = {d.PID}")
    print(f'  D0: {getD0(d)}')

Daughter: PT=72.11308288574219, Eta=-1.4164186716079712, Phi=-2.6831634044647217,  Charge=-1, PID = 11
  D0: 0.07568900002109077
Daughter: PT=22.022043228149414, Eta=-2.84055757522583, Phi=0.7728883624076843,  Charge=0, PID = 1000039
  D0: 0.24785018444524426
Daughter: PT=43.734039306640625, Eta=-2.298746347427368, Phi=1.9598586559295654,  Charge=1, PID = -11
  D0: 6.012322951318993
Daughter: PT=79.69889068603516, Eta=-2.1105077266693115, Phi=-0.040774766355752945,  Charge=0, PID = 1000039
  D0: 3.2992071515427424


In [8]:
for el in electrons:
    print(f"Electron: PT={el.PT}, Eta={el.Eta}, Phi={el.Phi}, Charge={el.Charge}, PID = {el.PID}")
    print(f'  D0: {el.D0}')

Electron: PT=71.72138214111328, Eta=-1.4164186716079712, Phi=-2.6831681728363037, Charge=-1, PID = 11
  D0: 0.07568765431642532
Electron: PT=44.11125946044922, Eta=-2.298746347427368, Phi=1.95987069606781, Charge=1, PID = -11
  D0: 6.012328147888184


In [11]:
print("ParticlePropagator")
for el in DelphesTree.ElectronNonIso:
    print(f"ElectronA: PT={el.PT}, Eta={el.Eta}, Phi={el.Phi}, Charge={el.Charge}")
    print(f'  D0: {el.D0}')

ParticlePropagator
ElectronA: PT=71.72138214111328, Eta=-1.4164186716079712, Phi=-2.6831681728363037, Charge=-1
  D0: 0.07568765431642532
ElectronA: PT=44.11125946044922, Eta=-2.298746347427368, Phi=1.95987069606781, Charge=1
  D0: 6.012328147888184
